# 36 — Ensemble pondéré des meilleures soumissions publiques

Réduction de variance pure : moyenne de rangs des meilleures soumissions, pondérée par leur marge au-dessus de 0.357.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from catboost import CatBoostClassifier
from src import config as C
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
te_op = op03_mask(test).to_numpy()
EPS = 1e-6; W = (5, 10, 20); SM = 30
def rowf(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def bb(df, ref):
    X = rowf(df).reset_index(drop=True)
    for c in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        X[f"fq_{c}"] = df[c].map(ref[c].value_counts(normalize=True)).fillna(0).values
    return pd.concat([X, behavioral_features(df, ref).reset_index(drop=True),
                      recency_features(df, ref).reset_index(drop=True),
                      recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, W).reset_index(drop=True)], axis=1)
def ftr(df, ref):
    X = bb(df, ref); X["te"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM); return X
def fap(df, ref):
    X = bb(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X
def cat(seed=42):
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=seed, verbose=False)
def rk(x):
    return np.argsort(np.argsort(x)) / (len(x) - 1)
def out(name, p_op):
    full = np.zeros(len(test)); full[te_op] = p_op
    print("écrit :", make_submission(test[C.ID], full, name))
ref0 = train.iloc[np.where(op03)[0]]; y0 = y_all[op03]
test_op = test.iloc[np.where(te_op)[0]].copy()

# Cache de prédictions partagé entre notebooks (évite de réentraîner les mêmes modèles)
ART = ROOT / "artifacts_preds"; ART.mkdir(exist_ok=True)
def cache(name, fn):
    f = ART / f"{name}.npy"
    if f.exists():
        print(f"cache : {f.name}"); return np.load(f)
    v = fn(); np.save(f, v); print(f"calculé + mis en cache : {f.name}"); return v
def train_champion(seed=42):
    return cat(seed).fit(ftr(ref0, ref0), y0).predict_proba(fap(test_op, ref0))[:, 1]
def pseudo_from(pch, thr_f=0.98, thr_l=0.02, seed=42):
    mfm = pch > thr_f; mlm = pch < thr_l
    pse = test_op.iloc[np.where(mfm | mlm)[0]].copy()
    pse[C.TARGET] = (pch[mfm | mlm] > thr_f).astype(float)
    aug = pd.concat([ref0, pse], ignore_index=True)
    print(f"pseudo ({thr_f}/{thr_l}, seed {seed}) : {int(mfm.sum())} fraudes / {int(mlm.sum())} légitimes")
    m = cat(seed).fit(ftr(aug, aug), aug[C.TARGET].to_numpy())
    return m.predict_proba(fap(test_op, aug))[:, 1]
print("setup OK |", op03.sum(), "op03 train /", te_op.sum(), "op03 test")

In [ ]:
S = ROOT / "submissions"
def load_op(fname):
    s = pd.read_csv(S / fname).set_index("id")["target"]
    return test_op[C.ID].map(s).fillna(0).to_numpy()
TOPS = {"28_rank_w50.csv": 0.357915, "28_rank_w52.csv": 0.357913, "28_rank_w60.csv": 0.357893,
        "28_rank_w65.csv": 0.357854, "26_rank_w70.csv": 0.357804, "23_pseudo_blend50.csv": 0.357396}
acc = np.zeros(int(te_op.sum())); wsum = 0.0
for f, score in TOPS.items():
    w = score - 0.3570
    acc += w * rk(load_op(f)); wsum += w
out("36_ensemble_top", acc / wsum)

## Résultat LB (mis à jour après soumission)

**0.357862** — sous le champion. Corr 0.9944 avec lui : nos bonnes soumissions sont le même modèle à des poids différents, l'ensemble n'ajoute rien.